In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python:", sys.version.split()[0])
print("Project root:", PROJECT_ROOT)

Python: 3.11.5
Project root: c:\RetailFlow


In [3]:
import pandas as pd
import numpy as np

from datetime import date, timedelta, datetime, timezone

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    DateType,
    StringType,
    DoubleType,
    IntegerType,
    StructType,
    StructField,
)

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Imports: PASSED")

Pandas: 2.3.1
NumPy: 2.3.1
Imports: PASSED


In [4]:
import os
from pathlib import Path

SPARK_TEMP_DIR = Path(r"C:\RetailFlow\spark_tmp")

SPARK_TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

os.environ["SPARK_LOCAL_DIRS"] = str(SPARK_TEMP_DIR)
os.environ["TEMP"] = str(SPARK_TEMP_DIR)
os.environ["TMP"] = str(SPARK_TEMP_DIR)
os.environ["TMPDIR"] = str(SPARK_TEMP_DIR)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Spark temporary directory:", SPARK_TEMP_DIR)
print("Directory exists:", SPARK_TEMP_DIR.exists())
print("Python executable:", sys.executable)
print("Spark environment: CONFIGURED")

Spark temporary directory: C:\RetailFlow\spark_tmp
Directory exists: True
Python executable: c:\RetailFlow\.venv\Scripts\python.exe
Spark environment: CONFIGURED


In [5]:
from pyspark.sql import SparkSession

from core.config import settings

SPARK_PACKAGES = (
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.4,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
    "io.delta:delta-spark_2.12:3.2.0"
)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("RetailFlow-Demand-Forecast-Persistence")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.local.dir", str(SPARK_TEMP_DIR))
    .config("spark.jars.packages", SPARK_PACKAGES)
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://localhost:9000"
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        settings.minio_access_key
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        settings.minio_secret_key
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    )
    .config(
        "spark.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )
    .config(
        "spark.delta.logStore.s3a.class",
        "io.delta.storage.S3SingleDriverLogStore"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Spark session: READY")

Spark version: 3.5.1
Master: local[*]
Spark session: READY


In [6]:
import os
from pathlib import Path

spark_temp_path = Path(spark.sparkContext._temp_dir)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("SparkContext temp:", spark_temp_path)
print("Temp exists:", spark_temp_path.exists())

test_df = spark.createDataFrame(
    [(1, "test")],
    ["id", "value"]
)

test_count = test_df.count()

print("Spark createDataFrame test rows:", test_count)

assert test_count == 1
assert SPARK_TEMP_DIR.exists()

print("Spark health check: PASSED")

Spark version: 3.5.1
Spark master: local[*]
SparkContext temp: C:\RetailFlow\spark_tmp\spark-bfdcfc25-3f18-4bc4-9bb7-7527b14b3571\pyspark-515ee485-e719-4a3a-96bf-bd39ab5c5030
Temp exists: True
Spark createDataFrame test rows: 1
Spark health check: PASSED


In [8]:
from core.constants import GOLD_FORECASTING_DATASET_PATH

FORECASTING_PATH = GOLD_FORECASTING_DATASET_PATH

forecasting_df = (
    spark.read
    .format("delta")
    .load(FORECASTING_PATH)
)

print("Forecasting dataset loaded")
print("Path:", FORECASTING_PATH)
print("Columns:", forecasting_df.columns)
print("Row count:", forecasting_df.count())

forecasting_df.printSchema()

Forecasting dataset loaded
Path: s3a://retailflow/gold/forecasting_dataset
Columns: ['date', 'store_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'supplier_id', 'currency', 'quantity_sold', 'revenue_cents', 'transactions', 'average_unit_price_cents', 'discount_cents', 'tax_cents', 'promotion_applied', 'promotion_transactions', 'inventory_min_before', 'inventory_end', 'day_of_week', 'day_of_month', 'month', 'year']
Row count: 9000
root
 |-- date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- quantity_sold: long (nullable = true)
 |-- revenue_cents: long (nullable = true)
 |-- transactions: long (nullable = true)
 |-- average_unit_price_cents: decimal(20,2) 

In [9]:
FORECAST_COLUMNS = [
    "date",
    "store_id",
    "product_id",
    "quantity_sold",
    "promotion_applied",
    "promotion_transactions",
    "inventory_min_before",
    "inventory_end",
    "average_unit_price_cents",
    "day_of_week",
    "day_of_month",
    "month",
    "year",
]

forecast_pdf = (
    forecasting_df
    .select(FORECAST_COLUMNS)
    .toPandas()
)

print("Pandas extraction completed")
print("Rows:", len(forecast_pdf))
print("Columns:", len(forecast_pdf.columns))
print("Memory usage MB:", round(
    forecast_pdf.memory_usage(deep=True).sum() / (1024 ** 2),
    2
))

print("Forecasting columns:")
print(list(forecast_pdf.columns))

Pandas extraction completed
Rows: 9000
Columns: 13
Memory usage MB: 2.84
Forecasting columns:
['date', 'store_id', 'product_id', 'quantity_sold', 'promotion_applied', 'promotion_transactions', 'inventory_min_before', 'inventory_end', 'average_unit_price_cents', 'day_of_week', 'day_of_month', 'month', 'year']


In [10]:
forecast_pdf["date"] = pd.to_datetime(
    forecast_pdf["date"],
    errors="raise"
).dt.date

forecast_pdf["quantity_sold"] = pd.to_numeric(
    forecast_pdf["quantity_sold"],
    errors="raise"
).astype(float)

forecast_pdf["promotion_applied"] = pd.to_numeric(
    forecast_pdf["promotion_applied"],
    errors="raise"
).astype(float)

forecast_pdf["promotion_transactions"] = pd.to_numeric(
    forecast_pdf["promotion_transactions"],
    errors="raise"
).astype(float)

forecast_pdf["inventory_min_before"] = pd.to_numeric(
    forecast_pdf["inventory_min_before"],
    errors="raise"
).astype(float)

forecast_pdf["inventory_end"] = pd.to_numeric(
    forecast_pdf["inventory_end"],
    errors="raise"
).astype(float)

forecast_pdf["average_unit_price_cents"] = pd.to_numeric(
    forecast_pdf["average_unit_price_cents"],
    errors="raise"
).astype(float)

forecast_pdf["day_of_week"] = pd.to_numeric(
    forecast_pdf["day_of_week"],
    errors="raise"
).astype(int)

forecast_pdf["day_of_month"] = pd.to_numeric(
    forecast_pdf["day_of_month"],
    errors="raise"
).astype(int)

forecast_pdf["month"] = pd.to_numeric(
    forecast_pdf["month"],
    errors="raise"
).astype(int)

forecast_pdf["year"] = pd.to_numeric(
    forecast_pdf["year"],
    errors="raise"
).astype(int)

duplicate_count = forecast_pdf.duplicated(
    subset=["date", "store_id", "product_id"]
).sum()

null_count = forecast_pdf[
    [
        "date",
        "store_id",
        "product_id",
        "quantity_sold",
    ]
].isnull().sum().sum()

negative_quantity = (
    forecast_pdf["quantity_sold"] < 0
).sum()

pair_count = forecast_pdf[
    ["store_id", "product_id"]
].drop_duplicates().shape[0]

date_count = forecast_pdf["date"].nunique()

min_date = forecast_pdf["date"].min()
max_date = forecast_pdf["date"].max()

print("Historical data validation")
print("Rows:", len(forecast_pdf))
print("Distinct dates:", date_count)
print("Store-product pairs:", pair_count)
print("Date range:", min_date, "->", max_date)
print("Duplicate grain rows:", duplicate_count)
print("Required-field nulls:", null_count)
print("Negative quantity rows:", negative_quantity)

assert len(forecast_pdf) == 9000
assert pair_count == 50
assert date_count == 180
assert duplicate_count == 0
assert null_count == 0
assert negative_quantity == 0
assert min_date == date(2026, 2, 14)
assert max_date == date(2026, 8, 12)

print("Historical data validation: PASSED")

Historical data validation
Rows: 9000
Distinct dates: 180
Store-product pairs: 50
Date range: 2026-02-14 -> 2026-08-12
Duplicate grain rows: 0
Required-field nulls: 0
Negative quantity rows: 0
Historical data validation: PASSED


In [11]:
forecast_pdf = forecast_pdf.sort_values(
    ["store_id", "product_id", "date"]
).reset_index(drop=True)

grouped = forecast_pdf.groupby(
    ["store_id", "product_id"],
    sort=False
)

forecast_pdf["lag_1"] = grouped["quantity_sold"].shift(1)
forecast_pdf["lag_7"] = grouped["quantity_sold"].shift(7)
forecast_pdf["lag_14"] = grouped["quantity_sold"].shift(14)
forecast_pdf["lag_28"] = grouped["quantity_sold"].shift(28)

forecast_pdf["rolling_mean_7"] = (
    grouped["quantity_sold"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=7,
            min_periods=7
        ).mean()
    )
)

forecast_pdf["rolling_mean_14"] = (
    grouped["quantity_sold"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=14,
            min_periods=14
        ).mean()
    )
)

forecast_pdf["rolling_mean_28"] = (
    grouped["quantity_sold"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=28,
            min_periods=28
        ).mean()
    )
)

MODEL_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
]

print("Feature engineering completed")
print("Rows:", len(forecast_pdf))
print("New features:")

for feature in MODEL_FEATURES:
    print(" -", feature)

Feature engineering completed
Rows: 9000
New features:
 - lag_1
 - lag_7
 - lag_14
 - lag_28
 - rolling_mean_7
 - rolling_mean_14
 - rolling_mean_28


In [12]:
expected_warmup_rows = 50 * 28

null_counts = forecast_pdf[MODEL_FEATURES].isnull().sum()

print("Feature validation")
print("Rows before warm-up removal:", len(forecast_pdf))
print()

print("Null counts:")

for feature in MODEL_FEATURES:
    print(f" - {feature}: {null_counts[feature]}")

total_feature_nulls = int(null_counts.sum())

print()
print("Total feature nulls:", total_feature_nulls)
print("Expected warm-up rows:", expected_warmup_rows)

assert null_counts["lag_1"] == 50
assert null_counts["lag_7"] == 350
assert null_counts["lag_14"] == 700
assert null_counts["lag_28"] == 1400

assert null_counts["rolling_mean_7"] == 350
assert null_counts["rolling_mean_14"] == 700
assert null_counts["rolling_mean_28"] == 1400

spot = (
    forecast_pdf[
        (forecast_pdf["store_id"] == "STR001") &
        (forecast_pdf["product_id"] == "PRD0001")
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

day_29 = spot.iloc[28]

manual_mean_7 = spot.loc[
    21:27,
    "quantity_sold"
].mean()

spark_style_mean = float(day_29["rolling_mean_7"])

print()
print("Rolling-window spot check")
print("Date:", day_29["date"])
print("Calculated rolling_mean_7:", round(spark_style_mean, 6))
print("Manual previous-7-day mean:", round(manual_mean_7, 6))

assert abs(
    spark_style_mean - manual_mean_7
) < 1e-6

print()
print("Rolling window leakage check: PASSED")

model_pdf = forecast_pdf.dropna(
    subset=MODEL_FEATURES
).copy()

model_pdf = model_pdf.reset_index(drop=True)

model_first_date = model_pdf["date"].min()
model_last_date = model_pdf["date"].max()

print()
print("Model-ready dataset")
print("Original rows:", len(forecast_pdf))
print("Model rows:", len(model_pdf))
print("Removed rows:", len(forecast_pdf) - len(model_pdf))
print("Model date range:", model_first_date, "->", model_last_date)

assert len(model_pdf) == 7600
assert len(forecast_pdf) - len(model_pdf) == 1400
assert model_first_date == date(2026, 3, 14)
assert model_last_date == date(2026, 8, 12)

print()
print("Feature validation: PASSED")
print("Model-ready dataset: PASSED")

Feature validation
Rows before warm-up removal: 9000

Null counts:
 - lag_1: 50
 - lag_7: 350
 - lag_14: 700
 - lag_28: 1400
 - rolling_mean_7: 350
 - rolling_mean_14: 700
 - rolling_mean_28: 1400

Total feature nulls: 4950
Expected warm-up rows: 1400

Rolling-window spot check
Date: 2026-03-14
Calculated rolling_mean_7: 11.285714
Manual previous-7-day mean: 11.285714

Rolling window leakage check: PASSED

Model-ready dataset
Original rows: 9000
Model rows: 7600
Removed rows: 1400
Model date range: 2026-03-14 -> 2026-08-12

Feature validation: PASSED
Model-ready dataset: PASSED


In [13]:
TEST_DAYS = 14

last_model_date = model_pdf["date"].max()

test_start = (
    last_model_date
    - timedelta(days=TEST_DAYS - 1)
)

train_pdf = model_pdf[
    model_pdf["date"] < test_start
].copy()

test_pdf = model_pdf[
    model_pdf["date"] >= test_start
].copy()

train_pdf = train_pdf.sort_values(
    ["date", "store_id", "product_id"]
).reset_index(drop=True)

test_pdf = test_pdf.sort_values(
    ["date", "store_id", "product_id"]
).reset_index(drop=True)

print("Chronological train/test split")
print("Last model date:", last_model_date)
print("Test starts:", test_start)

print()
print("Train rows:", len(train_pdf))
print("Test rows:", len(test_pdf))

print()
print(
    "Train range:",
    train_pdf["date"].min(),
    "->",
    train_pdf["date"].max()
)

print(
    "Test range:",
    test_pdf["date"].min(),
    "->",
    test_pdf["date"].max()
)

assert train_pdf["date"].max() < test_pdf["date"].min()

assert len(train_pdf) == 6900
assert len(test_pdf) == 700

assert test_pdf["date"].nunique() == 14
assert test_pdf["date"].min() == date(2026, 7, 30)
assert test_pdf["date"].max() == date(2026, 8, 12)

print()
print("Chronological split validation: PASSED")

Chronological train/test split
Last model date: 2026-08-12
Test starts: 2026-07-30

Train rows: 6900
Test rows: 700

Train range: 2026-03-14 -> 2026-07-29
Test range: 2026-07-30 -> 2026-08-12

Chronological split validation: PASSED


In [14]:
from sklearn.preprocessing import OneHotEncoder

CATEGORICAL_FEATURES = [
    "store_id",
    "product_id",
]

NUMERIC_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "promotion_applied",
    "promotion_transactions",
    "inventory_min_before",
    "average_unit_price_cents",
    "day_of_week",
    "day_of_month",
    "month",
]

TARGET_COLUMN = "quantity_sold"

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

train_categorical = encoder.fit_transform(
    train_pdf[CATEGORICAL_FEATURES]
)

test_categorical = encoder.transform(
    test_pdf[CATEGORICAL_FEATURES]
)

train_numeric = train_pdf[NUMERIC_FEATURES].to_numpy(
    dtype=np.float64
)

test_numeric = test_pdf[NUMERIC_FEATURES].to_numpy(
    dtype=np.float64
)

X_train = np.hstack([
    train_categorical,
    train_numeric,
])

X_test = np.hstack([
    test_categorical,
    test_numeric,
])

y_train = train_pdf[TARGET_COLUMN].to_numpy(
    dtype=np.float64
)

y_test = test_pdf[TARGET_COLUMN].to_numpy(
    dtype=np.float64
)

print("Training local Gradient Boosting model")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Encoded feature count:", X_train.shape[1])

local_gbt = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    loss="squared_error",
)

local_gbt.fit(
    X_train,
    y_train,
)

print("Local Gradient Boosting model trained successfully")

Training local Gradient Boosting model
Training rows: 6900
Test rows: 700
Encoded feature count: 29
Local Gradient Boosting model trained successfully


In [15]:
test_predictions = local_gbt.predict(X_test)

test_predictions = np.maximum(
    test_predictions,
    0.0
)

actual = y_test

mae = mean_absolute_error(
    actual,
    test_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        actual,
        test_predictions
    )
)

absolute_error = np.abs(
    actual - test_predictions
)

total_actual = np.sum(
    np.abs(actual)
)

wape = (
    np.sum(absolute_error) / total_actual * 100
    if total_actual > 0
    else 0.0
)

backtest_pdf = test_pdf[
    [
        "date",
        "store_id",
        "product_id",
        "quantity_sold",
    ]
].copy()

backtest_pdf["prediction"] = test_predictions
backtest_pdf["absolute_error"] = absolute_error

print("Historical backtest")
print("Test rows:", len(backtest_pdf))
print("Test dates:", backtest_pdf["date"].nunique())

print()
print("Local Gradient Boosting results")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"WAPE : {wape:.2f}%")

print()
print("Sample predictions")

print(
    backtest_pdf[
        [
            "date",
            "store_id",
            "product_id",
            "quantity_sold",
            "prediction",
            "absolute_error",
        ]
    ]
    .head(10)
    .to_string(index=False)
)
assert len(backtest_pdf) == 700
assert np.isfinite(mae)
assert np.isfinite(rmse)
assert np.isfinite(wape)
assert (backtest_pdf["prediction"] >= 0).all()

print()
print("Backtest validation: PASSED")

Historical backtest
Test rows: 700
Test dates: 14

Local Gradient Boosting results
MAE  : 1.0840
RMSE : 1.8324
WAPE : 29.13%

Sample predictions
      date store_id product_id  quantity_sold  prediction  absolute_error
2026-07-30   STR001    PRD0001           11.0    7.413463        3.586537
2026-07-30   STR001    PRD0002            8.0    6.603095        1.396905
2026-07-30   STR001    PRD0003            1.0    1.061400        0.061400
2026-07-30   STR001    PRD0004            1.0    1.109666        0.109666
2026-07-30   STR001    PRD0005            3.0    2.445944        0.554056
2026-07-30   STR001    PRD0006            2.0    3.456823        1.456823
2026-07-30   STR001    PRD0007            1.0    1.111771        0.111771
2026-07-30   STR001    PRD0008            1.0    1.063505        0.063505
2026-07-30   STR001    PRD0009            4.0    3.459836        0.540164
2026-07-30   STR001    PRD0010            7.0    5.868225        1.131775

Backtest validation: PASSED


In [16]:
baseline_predictions = test_pdf["lag_7"].to_numpy(
    dtype=np.float64
)

baseline_predictions = np.maximum(
    baseline_predictions,
    0.0
)

baseline_actual = y_test

baseline_mae = mean_absolute_error(
    baseline_actual,
    baseline_predictions
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        baseline_actual,
        baseline_predictions
    )
)

baseline_absolute_error = np.abs(
    baseline_actual - baseline_predictions
)

baseline_total_actual = np.sum(
    np.abs(baseline_actual)
)

baseline_wape = (
    np.sum(baseline_absolute_error)
    / baseline_total_actual
    * 100
    if baseline_total_actual > 0
    else 0.0
)

mae_improvement = baseline_mae - mae
rmse_improvement = baseline_rmse - rmse
wape_improvement = baseline_wape - wape

comparison_pdf = pd.DataFrame(
    [
        {
            "model": "Local Gradient Boosting",
            "MAE": mae,
            "RMSE": rmse,
            "WAPE_%": wape,
        },
        {
            "model": "7-Day Lag Baseline",
            "MAE": baseline_mae,
            "RMSE": baseline_rmse,
            "WAPE_%": baseline_wape,
        },
    ]
)

print("MODEL COMPARISON")
print(
    comparison_pdf.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print()
print("Improvement from Local Gradient Boosting")
print(f"MAE improvement  : {mae_improvement:+.4f}")
print(f"RMSE improvement : {rmse_improvement:+.4f}")
print(f"WAPE improvement : {wape_improvement:+.2f} percentage points")

model_beats_baseline = (
    mae < baseline_mae
    and rmse < baseline_rmse
    and wape < baseline_wape
)

print()

if model_beats_baseline:
    print("Local Gradient Boosting beats the 7-day baseline on all metrics.")
else:
    print("Local Gradient Boosting does not beat the baseline on all metrics.")

assert len(comparison_pdf) == 2
assert np.isfinite(baseline_mae)
assert np.isfinite(baseline_rmse)
assert np.isfinite(baseline_wape)

print("Model comparison validation: PASSED")

MODEL COMPARISON
                  model    MAE   RMSE  WAPE_%
Local Gradient Boosting 1.0840 1.8324 29.1279
     7-Day Lag Baseline 1.7286 3.0463 46.4491

Improvement from Local Gradient Boosting
MAE improvement  : +0.6446
RMSE improvement : +1.2139
WAPE improvement : +17.32 percentage points

Local Gradient Boosting beats the 7-day baseline on all metrics.
Model comparison validation: PASSED


In [17]:
RECURSIVE_HISTORY_DAYS = 28
FORECAST_DAYS = 7

FORECAST_START = max_date + timedelta(days=1)
FORECAST_END = (
    FORECAST_START
    + timedelta(days=FORECAST_DAYS - 1)
)

history_source = forecast_pdf[
    [
        "date",
        "store_id",
        "product_id",
        "quantity_sold",
        "promotion_applied",
        "promotion_transactions",
        "inventory_min_before",
        "inventory_end",
        "average_unit_price_cents",
        "day_of_week",
        "day_of_month",
        "month",
        "year",
    ]
].copy()

history_source = history_source.sort_values(
    ["store_id", "product_id", "date"]
).reset_index(drop=True)

forecast_entities = (
    history_source[
        ["store_id", "product_id"]
    ]
    .drop_duplicates()
    .sort_values(
        ["store_id", "product_id"]
    )
    .reset_index(drop=True)
)

print("Recursive forecast setup")
print("Last historical date:", max_date)
print("Forecast start:", FORECAST_START)
print("Forecast end:", FORECAST_END)
print("Forecast days:", FORECAST_DAYS)
print("Store-product pairs:", len(forecast_entities))

assert max_date == date(2026, 8, 12)
assert FORECAST_START == date(2026, 8, 13)
assert FORECAST_END == date(2026, 8, 19)
assert len(forecast_entities) == 50

history_counts = (
    history_source
    .groupby(
        ["store_id", "product_id"]
    )
    .size()
)

assert (history_counts >= RECURSIVE_HISTORY_DAYS).all()

print("Recursive history source: PASSED")

Recursive forecast setup
Last historical date: 2026-08-12
Forecast start: 2026-08-13
Forecast end: 2026-08-19
Forecast days: 7
Store-product pairs: 50
Recursive history source: PASSED


In [18]:
working_history = {}

for key, group in history_source.groupby(
    ["store_id", "product_id"],
    sort=False
):
    group = group.sort_values("date").reset_index(drop=True)

    working_history[key] = group.tail(
        RECURSIVE_HISTORY_DAYS
    ).copy()

forecast_records = []

for day_offset in range(FORECAST_DAYS):

    forecast_date = (
        FORECAST_START
        + timedelta(days=day_offset)
    )

    for store_id, product_id in forecast_entities[
        ["store_id", "product_id"]
    ].itertuples(index=False, name=None):

        key = (store_id, product_id)

        history = working_history[key].copy()
        history = history.sort_values("date").reset_index(drop=True)

        quantities = history["quantity_sold"].to_numpy(
            dtype=np.float64
        )

        lag_1 = quantities[-1]
        lag_7 = quantities[-7]
        lag_14 = quantities[-14]
        lag_28 = quantities[-28:]

        rolling_mean_7 = np.mean(
            quantities[-7:]
        )

        rolling_mean_14 = np.mean(
            quantities[-14:]
        )

        rolling_mean_28 = np.mean(
            quantities[-28:]
        )

        latest = history.iloc[-1]

        future_day_of_week = forecast_date.weekday() + 1
        future_day_of_month = forecast_date.day
        future_month = forecast_date.month
        future_year = forecast_date.year

        feature_row = pd.DataFrame(
            [
                {
                    "store_id": store_id,
                    "product_id": product_id,
                    "lag_1": lag_1,
                    "lag_7": lag_7,
                    "lag_14": lag_14,
                    "lag_28": quantities[-28],
                    "rolling_mean_7": rolling_mean_7,
                    "rolling_mean_14": rolling_mean_14,
                    "rolling_mean_28": rolling_mean_28,
                    "promotion_applied": float(
                        latest["promotion_applied"]
                    ),
                    "promotion_transactions": float(
                        latest["promotion_transactions"]
                    ),
                    "inventory_min_before": float(
                        latest["inventory_min_before"]
                    ),
                    "average_unit_price_cents": float(
                        latest["average_unit_price_cents"]
                    ),
                    "day_of_week": future_day_of_week,
                    "day_of_month": future_day_of_month,
                    "month": future_month,
                }
            ]
        )

        categorical_future = encoder.transform(
            feature_row[CATEGORICAL_FEATURES]
        )

        numeric_future = feature_row[
            NUMERIC_FEATURES
        ].to_numpy(
            dtype=np.float64
        )

        X_future = np.hstack(
            [
                categorical_future,
                numeric_future,
            ]
        )

        prediction = float(
            local_gbt.predict(X_future)[0]
        )

        prediction = max(
            prediction,
            0.0
        )

        forecast_records.append(
            {
                "date": forecast_date,
                "store_id": store_id,
                "product_id": product_id,
                "predicted_demand": prediction,
            }
        )

        new_history_row = {
            "date": forecast_date,
            "store_id": store_id,
            "product_id": product_id,
            "quantity_sold": prediction,
            "promotion_applied": float(
                latest["promotion_applied"]
            ),
            "promotion_transactions": float(
                latest["promotion_transactions"]
            ),
            "inventory_min_before": float(
                latest["inventory_min_before"]
            ),
            "inventory_end": float(
                latest["inventory_end"]
            ),
            "average_unit_price_cents": float(
                latest["average_unit_price_cents"]
            ),
            "day_of_week": future_day_of_week,
            "day_of_month": future_day_of_month,
            "month": future_month,
            "year": future_year,
        }

        working_history[key] = pd.concat(
            [
                history,
                pd.DataFrame([new_history_row]),
            ],
            ignore_index=True,
        ).tail(
            RECURSIVE_HISTORY_DAYS
        )

forecast_pdf_future = pd.DataFrame(
    forecast_records
)

forecast_pdf_future["predicted_demand"] = (
    forecast_pdf_future["predicted_demand"]
    .astype(float)
)

print("Recursive local ML forecast completed")
print("Forecast rows:", len(forecast_pdf_future))
print(
    "Forecast range:",
    forecast_pdf_future["date"].min(),
    "->",
    forecast_pdf_future["date"].max()
)

print()
print("Sample predictions")

print(
    forecast_pdf_future
    .head(10)
    .to_string(index=False)
)

Recursive local ML forecast completed
Forecast rows: 350
Forecast range: 2026-08-13 -> 2026-08-19

Sample predictions
      date store_id product_id  predicted_demand
2026-08-13   STR001    PRD0001          7.614519
2026-08-13   STR001    PRD0002          3.551759
2026-08-13   STR001    PRD0003          1.055328
2026-08-13   STR001    PRD0004          1.059142
2026-08-13   STR001    PRD0005          2.359336
2026-08-13   STR001    PRD0006          3.237939
2026-08-13   STR001    PRD0007          1.057434
2026-08-13   STR001    PRD0008          1.057434
2026-08-13   STR001    PRD0009          4.522892
2026-08-13   STR001    PRD0010          7.738084


In [19]:
forecast_pdf_future["date"] = pd.to_datetime(
    forecast_pdf_future["date"],
    errors="raise"
).dt.date

forecast_pdf_future["predicted_demand"] = pd.to_numeric(
    forecast_pdf_future["predicted_demand"],
    errors="raise"
).astype(float)

forecast_rows = len(forecast_pdf_future)

forecast_dates = forecast_pdf_future["date"].nunique()

forecast_pairs = forecast_pdf_future[
    ["store_id", "product_id"]
].drop_duplicates().shape[0]

forecast_grain = forecast_pdf_future[
    ["date", "store_id", "product_id"]
].drop_duplicates().shape[0]

negative_predictions = (
    forecast_pdf_future["predicted_demand"] < 0
).sum()

null_predictions = (
    forecast_pdf_future["predicted_demand"].isnull()
).sum()

min_forecast_date = forecast_pdf_future["date"].min()
max_forecast_date = forecast_pdf_future["date"].max()

print("Forecast validation")
print("Forecast rows:", forecast_rows)
print("Forecast dates:", forecast_dates)
print("Store-product pairs:", forecast_pairs)
print("Unique forecast grain rows:", forecast_grain)
print("Negative predictions:", negative_predictions)
print("Null predictions:", null_predictions)
print(
    "Forecast range:",
    min_forecast_date,
    "->",
    max_forecast_date
)

assert forecast_rows == 350
assert forecast_dates == 7
assert forecast_pairs == 50
assert forecast_grain == 350
assert negative_predictions == 0
assert null_predictions == 0

assert min_forecast_date == date(2026, 8, 13)
assert max_forecast_date == date(2026, 8, 19)

assert forecast_rows == (
    forecast_dates * forecast_pairs
)

print()
print("Forecast validation: PASSED")

Forecast validation
Forecast rows: 350
Forecast dates: 7
Store-product pairs: 50
Unique forecast grain rows: 350
Negative predictions: 0
Null predictions: 0
Forecast range: 2026-08-13 -> 2026-08-19

Forecast validation: PASSED


In [20]:
daily_forecast_summary = (
    forecast_pdf_future
    .groupby("date")
    .agg(
        active_pairs=("predicted_demand", "count"),
        total_predicted_units=("predicted_demand", "sum"),
        average_pair_demand=("predicted_demand", "mean"),
    )
    .reset_index()
)

daily_forecast_summary[
    "total_predicted_units"
] = daily_forecast_summary[
    "total_predicted_units"
].round(2)

daily_forecast_summary[
    "average_pair_demand"
] = daily_forecast_summary[
    "average_pair_demand"
].round(2)

print()
print("Daily forecast summary")
print(
    daily_forecast_summary.to_string(
        index=False
    )
)


Daily forecast summary
      date  active_pairs  total_predicted_units  average_pair_demand
2026-08-13            50                 173.40                 3.47
2026-08-14            50                 172.03                 3.44
2026-08-15            50                 182.40                 3.65
2026-08-16            50                 185.50                 3.71
2026-08-17            50                 174.53                 3.49
2026-08-18            50                 173.22                 3.46
2026-08-19            50                 172.23                 3.44


In [21]:
from pyspark.sql.types import (
    StructType,
    StructField,
    DateType,
    StringType,
    DoubleType,
)

forecast_schema = StructType([
    StructField("date", DateType(), False),
    StructField("store_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("predicted_demand", DoubleType(), False),
])

forecast_records = [
    (
        row["date"],
        str(row["store_id"]),
        str(row["product_id"]),
        float(row["predicted_demand"]),
    )
    for _, row in forecast_pdf_future.iterrows()
]

print("Preparing Spark DataFrame")
print("Records:", len(forecast_records))

forecast_spark_df = spark.createDataFrame(
    forecast_records,
    schema=forecast_schema,
)

spark_row_count = forecast_spark_df.count()

print("Spark DataFrame created")
print("Spark rows:", spark_row_count)

assert spark_row_count == 350

print("Spark forecast conversion: PASSED")

Preparing Spark DataFrame
Records: 350
Spark DataFrame created
Spark rows: 350
Spark forecast conversion: PASSED


In [22]:
try:
    import deltalake

    print("deltalake version:", deltalake.__version__)
    print("Delta-RS: AVAILABLE")

except ImportError:
    print("Delta-RS: NOT INSTALLED")

deltalake version: 1.6.2
Delta-RS: AVAILABLE


In [23]:
from deltalake import write_deltalake

print("Delta-RS imported successfully")
print("Forecast variable exists:", "forecast_pdf_future" in globals())

if "forecast_pdf_future" in globals():
    print("Forecast rows:", len(forecast_pdf_future))
    print(
        "Forecast range:",
        forecast_pdf_future["date"].min(),
        "->",
        forecast_pdf_future["date"].max()
    )

Delta-RS imported successfully
Forecast variable exists: True
Forecast rows: 350
Forecast range: 2026-08-13 -> 2026-08-19


In [24]:
import os

MINIO_ENDPOINT = "http://localhost:9000"
MINIO_BUCKET = "retailflow"
MINIO_FORECAST_PATH = "gold/demand_forecast"

storage_options = {
    "AWS_ACCESS_KEY_ID": settings.minio_access_key,
    "AWS_SECRET_ACCESS_KEY": settings.minio_secret_key,
    "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
    "AWS_REGION": "us-east-1",
    "AWS_ALLOW_HTTP": "true",
}

DELTA_FORECAST_URI = (
    f"s3://{MINIO_BUCKET}/{MINIO_FORECAST_PATH}"
)

print("Delta-RS storage configuration ready")
print("Endpoint:", MINIO_ENDPOINT)
print("Target:", DELTA_FORECAST_URI)
print(
    "Access key loaded:",
    bool(settings.minio_access_key)
)
print(
    "Secret key loaded:",
    bool(settings.minio_secret_key)
)

Delta-RS storage configuration ready
Endpoint: http://localhost:9000
Target: s3://retailflow/gold/demand_forecast
Access key loaded: True
Secret key loaded: True


In [25]:
from deltalake import write_deltalake

delta_forecast_pdf = forecast_pdf_future[
    [
        "date",
        "store_id",
        "product_id",
        "predicted_demand",
    ]
].copy()

delta_forecast_pdf["date"] = pd.to_datetime(
    delta_forecast_pdf["date"]
).dt.date

print("Writing forecast with Delta-RS")
print("Rows:", len(delta_forecast_pdf))
print("Target:", DELTA_FORECAST_URI)

write_deltalake(
    DELTA_FORECAST_URI,
    delta_forecast_pdf,
    mode="overwrite",
    storage_options=storage_options,
)

print("Delta-RS write completed")

Writing forecast with Delta-RS
Rows: 350
Target: s3://retailflow/gold/demand_forecast
Delta-RS write completed


In [26]:
from deltalake import DeltaTable

gold_forecast_table = DeltaTable(
    DELTA_FORECAST_URI,
    storage_options=storage_options,
)

gold_forecast_pdf = gold_forecast_table.to_pandas()

print("Gold demand forecast verification")
print("Rows:", len(gold_forecast_pdf))
print("Columns:", list(gold_forecast_pdf.columns))
print(
    "Date range:",
    gold_forecast_pdf["date"].min(),
    "->",
    gold_forecast_pdf["date"].max()
)

unique_dates = gold_forecast_pdf["date"].nunique()

unique_pairs = gold_forecast_pdf[
    ["store_id", "product_id"]
].drop_duplicates().shape[0]

unique_grain = gold_forecast_pdf[
    ["date", "store_id", "product_id"]
].drop_duplicates().shape[0]

negative_predictions = (
    gold_forecast_pdf["predicted_demand"] < 0
).sum()

print("Distinct dates:", unique_dates)
print("Store-product pairs:", unique_pairs)
print("Unique grain rows:", unique_grain)
print("Negative predictions:", negative_predictions)

assert len(gold_forecast_pdf) == 350
assert unique_dates == 7
assert unique_pairs == 50
assert unique_grain == 350
assert negative_predictions == 0

print("Gold demand forecast verification: PASSED")

Gold demand forecast verification
Rows: 350
Columns: ['date', 'store_id', 'product_id', 'predicted_demand']
Date range: 2026-08-13 -> 2026-08-19
Distinct dates: 7
Store-product pairs: 50
Unique grain rows: 350
Negative predictions: 0
Gold demand forecast verification: PASSED


In [27]:
from deltalake import DeltaTable

GOLD_INVENTORY_URI = "s3://retailflow/gold/inventory_current"

inventory_table = DeltaTable(
    GOLD_INVENTORY_URI,
    storage_options=storage_options,
)

inventory_pdf = inventory_table.to_pandas()

print("Inventory Gold loaded")
print("Rows:", len(inventory_pdf))
print("Columns:", list(inventory_pdf.columns))

print(
    "Store-product pairs:",
    inventory_pdf[
        ["store_id", "product_id"]
    ].drop_duplicates().shape[0]
)

Inventory Gold loaded
Rows: 50
Columns: ['store_id', 'store_name', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'supplier_id', 'inventory_quantity', 'last_transaction_timestamp', 'transaction_id', 'inventory_status']
Store-product pairs: 50


In [28]:
print("Inventory schema")
print(inventory_pdf.dtypes)

print()
print("Inventory sample")

print(
    inventory_pdf[
        [
            "store_id",
            "product_id",
            "product_name",
            "inventory_quantity",
            "last_transaction_timestamp",
            "inventory_status",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

Inventory schema
store_id                                   object
store_name                                 object
product_id                                 object
product_name                               object
category                                   object
subcategory                                object
brand                                      object
supplier_id                                object
inventory_quantity                          int32
last_transaction_timestamp    datetime64[us, UTC]
transaction_id                             object
inventory_status                           object
dtype: object

Inventory sample
store_id product_id             product_name  inventory_quantity       last_transaction_timestamp inventory_status
  STR001    PRD0001  Amul Full Cream Milk 1L                  83 2026-08-18 13:52:10.105632+00:00         IN_STOCK
  STR001    PRD0002 Fortune Basmati Rice 5kg                 113 2026-08-18 13:52:10.105632+00:00         IN_STOCK
  STR0

In [29]:
forecast_for_inventory = (
    gold_forecast_pdf[
        [
            "date",
            "store_id",
            "product_id",
            "predicted_demand",
        ]
    ]
    .copy()
)

inventory_for_risk = inventory_pdf[
    [
        "store_id",
        "store_name",
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "supplier_id",
        "inventory_quantity",
        "inventory_status",
    ]
].copy()

inventory_forecast = inventory_for_risk.merge(
    forecast_for_inventory,
    on=["store_id", "product_id"],
    how="left",
    validate="one_to_many",
)

print("Inventory-forecast join completed")
print("Rows:", len(inventory_forecast))
print(
    "Expected rows:",
    len(inventory_for_risk) * 7
)

print(
    "Missing forecast rows:",
    inventory_forecast["predicted_demand"]
    .isna()
    .sum()
)

assert len(inventory_forecast) == 350
assert inventory_forecast["predicted_demand"].notna().all()

print("Inventory-forecast join: PASSED")

Inventory-forecast join completed
Rows: 350
Expected rows: 350
Missing forecast rows: 0
Inventory-forecast join: PASSED


In [30]:
inventory_risk_base = (
    inventory_forecast
    .groupby(
        [
            "store_id",
            "store_name",
            "product_id",
            "product_name",
            "category",
            "subcategory",
            "brand",
            "supplier_id",
            "inventory_quantity",
            "inventory_status",
        ],
        as_index=False
    )
    .agg(
        forecast_7d_demand=(
            "predicted_demand",
            "sum"
        ),
        average_daily_demand=(
            "predicted_demand",
            "mean"
        ),
    )
)

inventory_risk_base["forecast_7d_demand"] = (
    inventory_risk_base["forecast_7d_demand"]
    .round(2)
)

inventory_risk_base["average_daily_demand"] = (
    inventory_risk_base["average_daily_demand"]
    .round(2)
)

inventory_risk_base["days_of_inventory"] = (
    inventory_risk_base["inventory_quantity"]
    / inventory_risk_base["average_daily_demand"]
    .replace(0, float("nan"))
)

inventory_risk_base["days_of_inventory"] = (
    inventory_risk_base["days_of_inventory"]
    .round(2)
)

print("Inventory coverage calculated")
print("Rows:", len(inventory_risk_base))

print()
print(
    inventory_risk_base[
        [
            "store_id",
            "product_id",
            "inventory_quantity",
            "forecast_7d_demand",
            "average_daily_demand",
            "days_of_inventory",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

assert len(inventory_risk_base) == 50

print()
print("Inventory coverage calculation: PASSED")

Inventory coverage calculated
Rows: 50

store_id product_id  inventory_quantity  forecast_7d_demand  average_daily_demand  days_of_inventory
  STR001    PRD0001                  83               58.38                  8.34               9.95
  STR001    PRD0002                 113               24.18                  3.45              32.75
  STR001    PRD0003                 176                7.43                  1.06             166.04
  STR001    PRD0004                 435                7.46                  1.07             406.54
  STR001    PRD0005                 213               16.93                  2.42              88.02
  STR001    PRD0006                 169               23.89                  3.41              49.56
  STR001    PRD0007                 270                7.46                  1.07             252.34
  STR001    PRD0008                 331                7.44                  1.06             312.26
  STR001    PRD0009                 448            

In [31]:
def classify_inventory_risk(days):
    if pd.isna(days):
        return "NO_DEMAND"
    if days < 3:
        return "STOCKOUT_RISK"
    if days < 7:
        return "HIGH_RISK"
    if days < 14:
        return "WATCH"
    if days <= 60:
        return "HEALTHY"
    return "OVERSTOCK"


inventory_risk_base["inventory_risk"] = (
    inventory_risk_base["days_of_inventory"]
    .apply(classify_inventory_risk)
)

print("Inventory risk classification")

print(
    inventory_risk_base[
        [
            "store_id",
            "product_id",
            "product_name",
            "inventory_quantity",
            "average_daily_demand",
            "days_of_inventory",
            "inventory_risk",
        ]
    ]
    .sort_values(
        "days_of_inventory",
        ascending=True
    )
    .head(15)
    .to_string(index=False)
)

print()
print("Risk distribution")

print(
    inventory_risk_base[
        "inventory_risk"
    ]
    .value_counts()
    .to_string()
)

print()
print("Inventory risk classification: PASSED")


Inventory risk classification
store_id product_id             product_name  inventory_quantity  average_daily_demand  days_of_inventory inventory_risk
  STR004    PRD0002 Fortune Basmati Rice 5kg                  21                  6.93               3.03      HIGH_RISK
  STR001    PRD0001  Amul Full Cream Milk 1L                  83                  8.34               9.95          WATCH
  STR004    PRD0009     Puma Running T-Shirt                  49                  2.91              16.84        HEALTHY
  STR004    PRD0010            Nestle KitKat                 127                  7.36              17.26        HEALTHY
  STR005    PRD0002 Fortune Basmati Rice 5kg                 144                  7.02              20.51        HEALTHY
  STR002    PRD0001  Amul Full Cream Milk 1L                 154                  7.17              21.48        HEALTHY
  STR004    PRD0005    Nike Air Zoom Pegasus                  45                  1.91              23.56        HEALTHY
  

In [32]:
risk_summary = (
    inventory_risk_base
    .groupby("inventory_risk")
    .agg(
        store_product_pairs=("product_id", "count"),
        total_inventory_units=("inventory_quantity", "sum"),
        total_7d_forecast=("forecast_7d_demand", "sum"),
        average_days_of_inventory=("days_of_inventory", "mean"),
    )
    .reset_index()
    .sort_values(
        "average_days_of_inventory",
        ascending=False
    )
)

risk_summary["average_days_of_inventory"] = (
    risk_summary["average_days_of_inventory"]
    .round(2)
)

print("Inventory risk exposure")
print(
    risk_summary.to_string(index=False)
)

print()
print("Total inventory units:",
      inventory_risk_base["inventory_quantity"].sum())

print(
    "Total forecast demand for 7 days:",
    round(
        inventory_risk_base["forecast_7d_demand"].sum(),
        2
    )
)

Inventory risk exposure
inventory_risk  store_product_pairs  total_inventory_units  total_7d_forecast  average_days_of_inventory
     OVERSTOCK                   27                   7304             368.72                     178.47
       HEALTHY                   21                   3987             757.64                      36.10
         WATCH                    1                     83              58.38                       9.95
     HIGH_RISK                    1                     21              48.52                       3.03

Total inventory units: 11395
Total forecast demand for 7 days: 1233.26


In [33]:
inventory_risk_base["projected_inventory_7d"] = (
    inventory_risk_base["inventory_quantity"]
    - inventory_risk_base["forecast_7d_demand"]
)

inventory_risk_base["projected_inventory_7d"] = (
    inventory_risk_base["projected_inventory_7d"]
    .round(2)
)

print("Projected 7-day inventory calculated")

print(
    inventory_risk_base[
        [
            "store_id",
            "product_id",
            "product_name",
            "inventory_quantity",
            "forecast_7d_demand",
            "projected_inventory_7d",
            "days_of_inventory",
            "inventory_risk",
        ]
    ]
    .sort_values(
        "projected_inventory_7d",
        ascending=True
    )
    .head(15)
    .to_string(index=False)
)

print()
print(
    "Projected stockout pairs:",
    (
        inventory_risk_base["projected_inventory_7d"] < 0
    ).sum()
)

print(
    "Projected zero-stock pairs:",
    (
        inventory_risk_base["projected_inventory_7d"] == 0
    ).sum()
)

print("Projected inventory calculation: PASSED")

Projected 7-day inventory calculated
store_id product_id             product_name  inventory_quantity  forecast_7d_demand  projected_inventory_7d  days_of_inventory inventory_risk
  STR004    PRD0002 Fortune Basmati Rice 5kg                  21               48.52                  -27.52               3.03      HIGH_RISK
  STR005    PRD0003          Apple iPhone 16                  27                7.45                   19.55              25.47        HEALTHY
  STR001    PRD0001  Amul Full Cream Milk 1L                  83               58.38                   24.62               9.95          WATCH
  STR004    PRD0009     Puma Running T-Shirt                  49               20.39                   28.61              16.84        HEALTHY
  STR004    PRD0005    Nike Air Zoom Pegasus                  45               13.35                   31.65              23.56        HEALTHY
  STR002    PRD0003          Apple iPhone 16                  45                7.78                   37

In [34]:
inventory_risk_base["recommended_reorder_quantity"] = (
    inventory_risk_base["forecast_7d_demand"]
    - inventory_risk_base["inventory_quantity"]
).clip(lower=0)

inventory_risk_base["recommended_reorder_quantity"] = (
    inventory_risk_base["recommended_reorder_quantity"]
    .apply(lambda x: int(np.ceil(x)))
)

print("Reorder quantity calculated")

print(
    inventory_risk_base[
        [
            "store_id",
            "product_id",
            "product_name",
            "inventory_quantity",
            "forecast_7d_demand",
            "projected_inventory_7d",
            "days_of_inventory",
            "inventory_risk",
            "recommended_reorder_quantity",
        ]
    ]
    .sort_values(
        "recommended_reorder_quantity",
        ascending=False
    )
    .head(15)
    .to_string(index=False)
)

print()
print(
    "Pairs requiring reorder:",
    (
        inventory_risk_base[
            "recommended_reorder_quantity"
        ] > 0
    ).sum()
)

print(
    "Total recommended reorder units:",
    inventory_risk_base[
        "recommended_reorder_quantity"
    ].sum()
)

print("Reorder calculation: PASSED")

Reorder quantity calculated
store_id product_id             product_name  inventory_quantity  forecast_7d_demand  projected_inventory_7d  days_of_inventory inventory_risk  recommended_reorder_quantity
  STR004    PRD0002 Fortune Basmati Rice 5kg                  21               48.52                  -27.52               3.03      HIGH_RISK                            28
  STR001    PRD0001  Amul Full Cream Milk 1L                  83               58.38                   24.62               9.95          WATCH                             0
  STR001    PRD0003          Apple iPhone 16                 176                7.43                  168.57             166.04      OVERSTOCK                             0
  STR001    PRD0004       Samsung Galaxy S25                 435                7.46                  427.54             406.54      OVERSTOCK                             0
  STR001    PRD0005    Nike Air Zoom Pegasus                 213               16.93                  196.0

In [35]:
inventory_risk_gold = inventory_risk_base[
    [
        "store_id",
        "store_name",
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "brand",
        "supplier_id",
        "inventory_quantity",
        "inventory_status",
        "forecast_7d_demand",
        "average_daily_demand",
        "days_of_inventory",
        "projected_inventory_7d",
        "inventory_risk",
        "recommended_reorder_quantity",
    ]
].copy()

print("Inventory risk Gold dataset prepared")
print("Rows:", len(inventory_risk_gold))
print("Columns:", list(inventory_risk_gold.columns))

print()
print(
    inventory_risk_gold
    .sort_values(
        "recommended_reorder_quantity",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)

assert len(inventory_risk_gold) == 50

assert (
    inventory_risk_gold[
        [
            "store_id",
            "product_id",
        ]
    ]
    .drop_duplicates()
    .shape[0]
    == 50
)

assert (
    inventory_risk_gold[
        "recommended_reorder_quantity"
    ].min()
    >= 0
)

print()
print("Inventory risk Gold preparation: PASSED")

Inventory risk Gold dataset prepared
Rows: 50
Columns: ['store_id', 'store_name', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'supplier_id', 'inventory_quantity', 'inventory_status', 'forecast_7d_demand', 'average_daily_demand', 'days_of_inventory', 'projected_inventory_7d', 'inventory_risk', 'recommended_reorder_quantity']

store_id                 store_name product_id             product_name    category subcategory   brand supplier_id  inventory_quantity inventory_status  forecast_7d_demand  average_daily_demand  days_of_inventory  projected_inventory_7d inventory_risk  recommended_reorder_quantity
  STR004       RetailFlow Salt Lake    PRD0002 Fortune Basmati Rice 5kg   Groceries        Rice Fortune      SUP002                  21         IN_STOCK               48.52                  6.93               3.03                  -27.52      HIGH_RISK                            28
  STR001 RetailFlow Connaught Place    PRD0001  Amul Full Cream Milk 1L   Groceries  

In [36]:
GOLD_INVENTORY_RISK_URI = (
    "s3://retailflow/gold/inventory_risk"
)

print("Writing inventory risk to Gold")
print("Rows:", len(inventory_risk_gold))
print("Target:", GOLD_INVENTORY_RISK_URI)

write_deltalake(
    GOLD_INVENTORY_RISK_URI,
    inventory_risk_gold,
    mode="overwrite",
    storage_options=storage_options,
)

print("Inventory risk Gold write completed")

Writing inventory risk to Gold
Rows: 50
Target: s3://retailflow/gold/inventory_risk
Inventory risk Gold write completed


In [37]:
inventory_risk_table = DeltaTable(
    GOLD_INVENTORY_RISK_URI,
    storage_options=storage_options,
)

inventory_risk_verified = (
    inventory_risk_table
    .to_pandas()
)

print("Inventory risk Gold verification")
print("Rows:", len(inventory_risk_verified))
print(
    "Columns:",
    list(inventory_risk_verified.columns)
)

print()
print(
    inventory_risk_verified[
        [
            "store_id",
            "product_id",
            "product_name",
            "inventory_quantity",
            "forecast_7d_demand",
            "days_of_inventory",
            "projected_inventory_7d",
            "inventory_risk",
            "recommended_reorder_quantity",
        ]
    ]
    .sort_values(
        "recommended_reorder_quantity",
        ascending=False
    )
    .head(10)
    .to_string(index=False)
)

assert len(inventory_risk_verified) == 50

assert (
    inventory_risk_verified[
        ["store_id", "product_id"]
    ]
    .drop_duplicates()
    .shape[0]
    == 50
)

assert (
    inventory_risk_verified[
        "recommended_reorder_quantity"
    ].min()
    >= 0
)

print()
print("Inventory risk Gold verification: PASSED")

Inventory risk Gold verification
Rows: 50
Columns: ['store_id', 'store_name', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'supplier_id', 'inventory_quantity', 'inventory_status', 'forecast_7d_demand', 'average_daily_demand', 'days_of_inventory', 'projected_inventory_7d', 'inventory_risk', 'recommended_reorder_quantity']

store_id product_id             product_name  inventory_quantity  forecast_7d_demand  days_of_inventory  projected_inventory_7d inventory_risk  recommended_reorder_quantity
  STR004    PRD0002 Fortune Basmati Rice 5kg                  21               48.52               3.03                  -27.52      HIGH_RISK                            28
  STR001    PRD0001  Amul Full Cream Milk 1L                  83               58.38               9.95                   24.62          WATCH                             0
  STR001    PRD0003          Apple iPhone 16                 176                7.43             166.04                  168.57      OVE

In [38]:
GOLD_FINANCIAL_SUMMARY_URI = (
    "s3://retailflow/gold/financial_summary"
)

GOLD_PAYMENT_FINANCE_URI = (
    "s3://retailflow/gold/payment_finance"
)

GOLD_DAILY_SALES_URI = (
    "s3://retailflow/gold/daily_sales"
)

financial_summary_pdf = (
    DeltaTable(
        GOLD_FINANCIAL_SUMMARY_URI,
        storage_options=storage_options,
    )
    .to_pandas()
)

payment_finance_pdf = (
    DeltaTable(
        GOLD_PAYMENT_FINANCE_URI,
        storage_options=storage_options,
    )
    .to_pandas()
)

daily_sales_pdf = (
    DeltaTable(
        GOLD_DAILY_SALES_URI,
        storage_options=storage_options,
    )
    .to_pandas()
)

print("Financial Gold datasets loaded")

print(
    "Financial summary rows:",
    len(financial_summary_pdf)
)

print(
    "Payment finance rows:",
    len(payment_finance_pdf)
)

print(
    "Daily sales rows:",
    len(daily_sales_pdf)
)

Financial Gold datasets loaded
Financial summary rows: 1
Payment finance rows: 6
Daily sales rows: 14433


In [39]:
print("Financial summary")
print(financial_summary_pdf.dtypes)
print()

print("Payment finance")
print(payment_finance_pdf.dtypes)
print()

print("Daily sales")
print(daily_sales_pdf.dtypes)

Financial summary
total_revenue                  int64
total_tax                      int64
total_discount                 int64
average_transaction_value    float64
dtype: object

Payment finance
payment_method    object
revenue            int64
tax                int64
discount           int64
dtype: object

Daily sales
transaction_timestamp     object
revenue                    int64
transactions               int64
average_order_value      float64
dtype: object


In [41]:
daily_sales_check = daily_sales_pdf.copy()

daily_sales_check["transaction_timestamp"] = pd.to_datetime(
    daily_sales_check["transaction_timestamp"],
    format="mixed",
    errors="raise"
)

daily_sales_check["date"] = (
    daily_sales_check["transaction_timestamp"].dt.date
)

print("Daily sales grain check")
print("Rows:", len(daily_sales_check))
print("Distinct dates:", daily_sales_check["date"].nunique())

print()
print("Rows per date:")

print(
    daily_sales_check
    .groupby("date")
    .size()
    .describe()
    .to_string()
)

print()
print(
    daily_sales_check[
        [
            "transaction_timestamp",
            "revenue",
            "transactions",
            "average_order_value",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

Daily sales grain check
Rows: 14433
Distinct dates: 182

Rows per date:
count    182.000000
mean      79.302198
std       10.733944
min        5.000000
25%       76.000000
50%       79.000000
75%       83.000000
max      155.000000

     transaction_timestamp  revenue  transactions  average_order_value
2026-07-23 10:16:53.651507    12600             1              12600.0
2026-02-14 20:09:59.975050   212625             1             212625.0
2026-06-09 11:52:45.092242    15750             1              15750.0
2026-03-07 19:43:53.751371     6300             1               6300.0
2026-08-11 19:03:12.073111  8848820             1            8848820.0
2026-03-01 17:52:46.345620  3184938             1            3184938.0
2026-05-04 12:16:04.323945  8848820             1            8848820.0
2026-04-27 19:18:08.523888  5917599             1            5917599.0
2026-06-29 16:47:45.221467   335776             1             335776.0
2026-03-11 11:06:17.083487  3538820             1        

In [42]:
daily_financials = (
    daily_sales_check
    .groupby("date", as_index=False)
    .agg(
        revenue=("revenue", "sum"),
        transactions=("transactions", "sum"),
    )
)

daily_financials["average_order_value"] = (
    daily_financials["revenue"]
    / daily_financials["transactions"]
).round(2)

daily_financials = daily_financials.sort_values(
    "date"
).reset_index(drop=True)

print("Daily financial aggregation completed")
print("Rows:", len(daily_financials))
print(
    "Date range:",
    daily_financials["date"].min(),
    "->",
    daily_financials["date"].max()
)

print()
print(
    daily_financials.head(10)
    .to_string(index=False)
)

assert len(daily_financials) == 182
assert daily_financials["transactions"].min() > 0
assert daily_financials["revenue"].min() >= 0

print()
print("Daily financial aggregation: PASSED")

Daily financial aggregation completed
Rows: 182
Date range: 2026-02-14 -> 2026-08-18

      date   revenue  transactions  average_order_value
2026-02-14 192800068            75           2570667.57
2026-02-15 172701620            83           2080742.41
2026-02-16 165607295            80           2070091.19
2026-02-17 158795915            80           1984948.94
2026-02-18 166492183            79           2107495.99
2026-02-19 168149271            79           2128471.78
2026-02-20 165503750            80           2068796.88
2026-02-21 199655949            82           2434828.65
2026-02-22 156641860            70           2237740.86
2026-02-23 158816009            78           2036102.68

Daily financial aggregation: PASSED


In [43]:
daily_total_revenue = int(
    daily_financials["revenue"].sum()
)

daily_total_transactions = int(
    daily_financials["transactions"].sum()
)

summary_total_revenue = int(
    financial_summary_pdf.loc[
        0,
        "total_revenue"
    ]
)

summary_average_transaction_value = float(
    financial_summary_pdf.loc[
        0,
        "average_transaction_value"
    ]
)

calculated_average_transaction_value = (
    daily_total_revenue
    / daily_total_transactions
)

print("Financial reconciliation")
print("Daily total revenue:", daily_total_revenue)
print("Summary total revenue:", summary_total_revenue)
print(
    "Revenue difference:",
    daily_total_revenue - summary_total_revenue
)

print()
print(
    "Daily transaction count:",
    daily_total_transactions
)

print(
    "Calculated average transaction value:",
    round(
        calculated_average_transaction_value,
        2
    )
)

print(
    "Summary average transaction value:",
    round(
        summary_average_transaction_value,
        2
    )
)

revenue_matches = (
    daily_total_revenue == summary_total_revenue
)

print()
print(
    "Revenue reconciliation:",
    "PASSED" if revenue_matches else "FAILED"
)

Financial reconciliation
Daily total revenue: 79115735712
Summary total revenue: 79115735712
Revenue difference: 0

Daily transaction count: 20217
Calculated average transaction value: 3913327.19
Summary average transaction value: 3913327.19

Revenue reconciliation: PASSED


In [45]:
GOLD_DAILY_SALES_URI = "s3://retailflow/gold/daily_sales"

print("Replacing daily sales Gold table")
print("Rows:", len(daily_financials))
print("Target:", GOLD_DAILY_SALES_URI)

write_deltalake(
    GOLD_DAILY_SALES_URI,
    daily_financials,
    mode="overwrite",
    schema_mode="overwrite",
    storage_options=storage_options,
)

print("Corrected daily sales write completed")

Replacing daily sales Gold table
Rows: 182
Target: s3://retailflow/gold/daily_sales
Corrected daily sales write completed


In [46]:
daily_sales_verified = (
    DeltaTable(
        GOLD_DAILY_SALES_URI,
        storage_options=storage_options,
    )
    .to_pandas()
)

daily_sales_verified["date"] = pd.to_datetime(
    daily_sales_verified["date"]
).dt.date

verified_revenue = int(
    daily_sales_verified["revenue"].sum()
)

verified_transactions = int(
    daily_sales_verified["transactions"].sum()
)

print("Corrected daily sales verification")
print("Rows:", len(daily_sales_verified))
print("Distinct dates:", daily_sales_verified["date"].nunique())
print(
    "Date range:",
    daily_sales_verified["date"].min(),
    "->",
    daily_sales_verified["date"].max()
)

print("Total revenue:", verified_revenue)
print("Total transactions:", verified_transactions)

print(
    "Revenue difference:",
    verified_revenue - summary_total_revenue
)

print(
    "Transaction difference:",
    verified_transactions - daily_total_transactions
)

assert len(daily_sales_verified) == 182
assert daily_sales_verified["date"].nunique() == 182
assert verified_revenue == summary_total_revenue
assert verified_transactions == daily_total_transactions

print("Daily sales Gold verification: PASSED")

Corrected daily sales verification
Rows: 182
Distinct dates: 182
Date range: 2026-02-14 -> 2026-08-18
Total revenue: 79115735712
Total transactions: 20217
Revenue difference: 0
Transaction difference: 0
Daily sales Gold verification: PASSED


In [47]:
payment_total_revenue = int(
    payment_finance_pdf["revenue"].sum()
)

payment_total_tax = int(
    payment_finance_pdf["tax"].sum()
)

payment_total_discount = int(
    payment_finance_pdf["discount"].sum()
)

summary_total_tax = int(
    financial_summary_pdf.loc[0, "total_tax"]
)

summary_total_discount = int(
    financial_summary_pdf.loc[0, "total_discount"]
)

print("Payment finance reconciliation")
print("Payment revenue:", payment_total_revenue)
print("Financial summary revenue:", summary_total_revenue)
print("Revenue difference:", payment_total_revenue - summary_total_revenue)

print()
print("Payment tax:", payment_total_tax)
print("Financial summary tax:", summary_total_tax)
print("Tax difference:", payment_total_tax - summary_total_tax)

print()
print("Payment discount:", payment_total_discount)
print("Financial summary discount:", summary_total_discount)
print(
    "Discount difference:",
    payment_total_discount - summary_total_discount
)

assert payment_total_revenue == summary_total_revenue
assert payment_total_tax == summary_total_tax
assert payment_total_discount == summary_total_discount

print()
print("Payment finance reconciliation: PASSED")

Payment finance reconciliation
Payment revenue: 79115735712
Financial summary revenue: 79115735712
Revenue difference: 0

Payment tax: 11700538212
Financial summary tax: 11700538212
Tax difference: 0

Payment discount: 11983814100
Financial summary discount: 11983814100
Discount difference: 0

Payment finance reconciliation: PASSED


In [48]:
print("Financial Gold final audit")

print()
print("Financial summary")
print("Rows:", len(financial_summary_pdf))
print(
    "Revenue:",
    financial_summary_pdf.loc[0, "total_revenue"]
)
print(
    "Tax:",
    financial_summary_pdf.loc[0, "total_tax"]
)
print(
    "Discount:",
    financial_summary_pdf.loc[0, "total_discount"]
)

print()
print("Payment finance")
print("Rows:", len(payment_finance_pdf))
print(
    "Payment methods:",
    payment_finance_pdf["payment_method"].tolist()
)

print()
print("Daily sales")
print("Rows:", len(daily_sales_verified))
print("Distinct dates:", daily_sales_verified["date"].nunique())
print(
    "Revenue:",
    daily_sales_verified["revenue"].sum()
)
print(
    "Transactions:",
    daily_sales_verified["transactions"].sum()
)

assert len(financial_summary_pdf) == 1
assert len(payment_finance_pdf) == 6
assert len(daily_sales_verified) == 182

assert (
    daily_sales_verified["revenue"].sum()
    == financial_summary_pdf.loc[0, "total_revenue"]
)

assert (
    payment_finance_pdf["revenue"].sum()
    == financial_summary_pdf.loc[0, "total_revenue"]
)

assert (
    payment_finance_pdf["tax"].sum()
    == financial_summary_pdf.loc[0, "total_tax"]
)

assert (
    payment_finance_pdf["discount"].sum()
    == financial_summary_pdf.loc[0, "total_discount"]
)

print()
print("Financial Gold final audit: PASSED")

Financial Gold final audit

Financial summary
Rows: 1
Revenue: 79115735712
Tax: 11700538212
Discount: 11983814100

Payment finance
Rows: 6
Payment methods: ['Credit Card', 'Bank Transfer', 'Cash', 'Debit Card', 'Wallet', 'UPI']

Daily sales
Rows: 182
Distinct dates: 182
Revenue: 79115735712
Transactions: 20217

Financial Gold final audit: PASSED


In [49]:
GOLD_DATASETS = {
    "category_sales": "s3://retailflow/gold/category_sales",
    "city_sales": "s3://retailflow/gold/city_sales",
    "store_sales": "s3://retailflow/gold/store_sales",
    "top_products": "s3://retailflow/gold/top_products",
    "payment_summary": "s3://retailflow/gold/payment_summary",
    "loyalty_analysis": "s3://retailflow/gold/loyalty_analysis",
    "customer_segments": "s3://retailflow/gold/customer_segments",
}

dashboard_gold = {}

for name, uri in GOLD_DATASETS.items():
    table = DeltaTable(
        uri,
        storage_options=storage_options,
    )

    df = table.to_pandas()
    dashboard_gold[name] = df

    print(
        f"{name}: {len(df)} rows | "
        f"{len(df.columns)} columns"
    )

print()
print("Dashboard Gold discovery: COMPLETED")

category_sales: 4 rows | 3 columns
city_sales: 5 rows | 3 columns
store_sales: 5 rows | 5 columns
top_products: 10 rows | 4 columns
payment_summary: 6 rows | 3 columns
loyalty_analysis: 2 rows | 4 columns
customer_segments: 3 rows | 4 columns

Dashboard Gold discovery: COMPLETED


In [ ]:
print("Dashboard Gold schema audit")

for name, df in dashboard_gold.items():
    print()
    print(name)
    print("-" * len(name))
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print(df.dtypes.to_string())

print()
print("Dashboard schema audit: COMPLETED")

Dashboard Gold schema audit

category_sales
--------------
Rows: 4
Columns: ['category', 'revenue', 'orders']
category    object
revenue      int64
orders       int64

city_sales
----------
Rows: 5
Columns: ['city', 'revenue', 'transactions']
city            object
revenue          int64
transactions     int64

store_sales
-----------
Rows: 5
Columns: ['store_id', 'store_name', 'revenue', 'transactions', 'average_sale']
store_id         object
store_name       object
revenue           int64
transactions      int64
average_sale    float64

top_products
------------
Rows: 10
Columns: ['product_id', 'product_name', 'quantity_sold', 'revenue']
product_id       object
product_name     object
quantity_sold     int64
revenue           int64

payment_summary
---------------
Rows: 6
Columns: ['payment_method', 'revenue', 'transactions']
payment_method    object
revenue            int64
transactions       int64

loyalty_analysis
----------------
Rows: 2
Columns: ['loyalty_member', 'transactions'

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 50505)
Traceback (most recent call last):
  File "C:\Users\vinee\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "C:\Users\vinee\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "C:\Users\vinee\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "C:\Users\vinee\AppData\Local\Programs\Python\Python311\Lib\socketserver.py", line 755, in __init__
    self.handle()
  File "c:\RetailFlow\.venv\Lib\site-packages\pyspark\accumulators.py", line 295, in handle
    poll(accum_updates)
  File "c:\RetailFlow\.venv\Lib\site-packages\pyspark\accumulators.py", line 267, in poll
    if self

: 